# Notebook 09 — Gene–Protein–Enzyme–Reaction Network

## Integración mecanística de la firma metabólica validada en Parkinson

### Objetivo
Convertir los resultados priorizados del Notebook 08 en una red interpretable que conecte:

**gen → proteína → enzima/complejo → reacción → metabolito → tema metabólico**

### Principios de interpretación
- La red representa relaciones bioquímicas y de anotación, no causalidad.
- Un cambio en RNA no implica directamente cambio de proteína.
- Un cambio en proteína no implica directamente cambio de actividad enzimática.
- La red no mide flujo metabólico.
- Las subunidades de complejos se etiquetan como `complex-associated`.

## 0. Imports, configuración y rutas

In [ ]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

RANDOM_STATE = 42

cwd = Path.cwd()

if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = Path(r"C:\Users\Camfe\pythonlab")

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"

NB09_RESULTS = RESULTS_DIR / "networks" / "notebook09"
NB09_FIGURES = FIGURES_DIR / "networks" / "notebook09"

NB09_RESULTS.mkdir(parents=True, exist_ok=True)
NB09_FIGURES.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("NB09_RESULTS:", NB09_RESULTS)
print("NB09_FIGURES:", NB09_FIGURES)

## 1. Localizar outputs necesarios

In [ ]:
NETWORK_INPUT = (
    PROJECT_ROOT
    / "results"
    / "validation"
    / "notebook08"
    / "Notebook09_gene_network_candidates.csv"
)

NB08_EVIDENCE = (
    PROJECT_ROOT
    / "results"
    / "validation"
    / "notebook08"
    / "cross_dataset_evidence_table.csv"
)

NB08_MANIFEST = (
    PROJECT_ROOT
    / "results"
    / "validation"
    / "notebook08"
    / "notebook08_manifest.json"
)

def fallback_find(path, pattern):
    if path.exists():
        return path
    matches = list(PROJECT_ROOT.rglob(pattern))
    return matches[0] if matches else path

NETWORK_INPUT = fallback_find(
    NETWORK_INPUT,
    "Notebook09_gene_network_candidates.csv",
)

NB08_EVIDENCE = fallback_find(
    NB08_EVIDENCE,
    "cross_dataset_evidence_table.csv",
)

NB08_MANIFEST = fallback_find(
    NB08_MANIFEST,
    "notebook08_manifest.json",
)

for name, path in {
    "NETWORK_INPUT": NETWORK_INPUT,
    "NB08_EVIDENCE": NB08_EVIDENCE,
    "NB08_MANIFEST": NB08_MANIFEST,
}.items():
    print(name, "->", path, "| exists:", path.exists())

## 2. Verificar archivo esencial

In [ ]:
if not NETWORK_INPUT.exists():
    raise FileNotFoundError(
        f"Notebook 08 network input not found:\n{NETWORK_INPUT}"
    )

print("✓ Notebook 08 network input found.")

## 3. Cargar genes priorizados del Notebook 08

In [ ]:
candidates = pd.read_csv(NETWORK_INPUT)

print("Shape:", candidates.shape)
print("Columns:")
print(candidates.columns.tolist())

required = {
    "signature_gene",
    "cross_dataset_score",
    "final_validation_class",
}

missing = required.difference(candidates.columns)

if missing:
    raise KeyError(
        f"Network candidate table is missing: {sorted(missing)}"
    )

candidates["signature_gene"] = (
    candidates["signature_gene"]
    .astype(str)
    .str.strip()
)

candidates["gene_upper"] = (
    candidates["signature_gene"]
    .str.upper()
)

candidates["cross_dataset_score"] = pd.to_numeric(
    candidates["cross_dataset_score"],
    errors="coerce",
)

if candidates["cross_dataset_score"].isna().any():
    raise ValueError(
        "NaN cross-dataset scores found. Review Notebook 08."
    )

display(candidates)

## 4. Definir conjunto de genes para la red

In [ ]:
network_genes = (
    candidates["signature_gene"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print("Genes entering network:", len(network_genes))
print(network_genes)

if len(network_genes) < 3:
    raise RuntimeError(
        "Too few genes for a meaningful network."
    )

## 5. Anotación bioquímica curada

In [ ]:
curated_records = [
    {
        "gene": "GPI",
        "protein": "Glucose-6-phosphate isomerase",
        "enzyme_role": "enzyme",
        "ec": "5.3.1.9",
        "reaction": "G6P_to_F6P",
        "reaction_label": "Glucose-6-phosphate ↔ Fructose-6-phosphate",
        "substrates": ["Glucose-6-phosphate"],
        "products": ["Fructose-6-phosphate"],
        "themes": ["Glycolysis/Glucose"],
    },
    {
        "gene": "MDH2",
        "protein": "Malate dehydrogenase, mitochondrial",
        "enzyme_role": "enzyme",
        "ec": "1.1.1.37",
        "reaction": "Malate_to_Oxaloacetate",
        "reaction_label": "Malate ↔ Oxaloacetate",
        "substrates": ["Malate"],
        "products": ["Oxaloacetate"],
        "themes": ["TCA/Pyruvate", "Redox"],
    },
    {
        "gene": "ALDOC",
        "protein": "Fructose-bisphosphate aldolase C",
        "enzyme_role": "enzyme",
        "ec": "4.1.2.13",
        "reaction": "F16BP_to_GAP_DHAP",
        "reaction_label": "Fructose-1,6-bisphosphate ↔ GAP + DHAP",
        "substrates": ["Fructose-1,6-bisphosphate"],
        "products": ["Glyceraldehyde-3-phosphate", "Dihydroxyacetone phosphate"],
        "themes": ["Glycolysis/Glucose"],
    },
    {
        "gene": "TH",
        "protein": "Tyrosine hydroxylase",
        "enzyme_role": "enzyme",
        "ec": "1.14.16.2",
        "reaction": "Tyrosine_to_LDOPA",
        "reaction_label": "L-Tyrosine → L-DOPA",
        "substrates": ["L-Tyrosine"],
        "products": ["L-DOPA"],
        "themes": ["Dopamine/Catecholamine"],
    },
    {
        "gene": "DDC",
        "protein": "Aromatic-L-amino-acid decarboxylase",
        "enzyme_role": "enzyme",
        "ec": "4.1.1.28",
        "reaction": "LDOPA_to_Dopamine",
        "reaction_label": "L-DOPA → Dopamine",
        "substrates": ["L-DOPA"],
        "products": ["Dopamine"],
        "themes": ["Dopamine/Catecholamine"],
    },
    {
        "gene": "FH",
        "protein": "Fumarate hydratase, mitochondrial",
        "enzyme_role": "enzyme",
        "ec": "4.2.1.2",
        "reaction": "Fumarate_to_Malate",
        "reaction_label": "Fumarate ↔ Malate",
        "substrates": ["Fumarate"],
        "products": ["Malate"],
        "themes": ["TCA/Pyruvate"],
    },
    {
        "gene": "NDUFS5",
        "protein": "NADH dehydrogenase [ubiquinone] iron-sulfur protein 5",
        "enzyme_role": "complex-associated",
        "ec": "7.1.1.2",
        "reaction": "Complex_I_NADH_Ubiquinone",
        "reaction_label": "NADH + Ubiquinone → NAD+ + Ubiquinol (Complex I)",
        "substrates": ["NADH", "Ubiquinone"],
        "products": ["NAD+", "Ubiquinol"],
        "themes": ["Mito/OXPHOS", "Redox"],
    },
    {
        "gene": "ALDH1A1",
        "protein": "Aldehyde dehydrogenase 1 family member A1",
        "enzyme_role": "enzyme",
        "ec": "1.2.1.36",
        "reaction": "Retinal_to_Retinoic_acid",
        "reaction_label": "Retinal → Retinoic acid",
        "substrates": ["Retinal"],
        "products": ["Retinoic acid"],
        "themes": ["Redox", "General metabolism"],
    },
    {
        "gene": "ACO2",
        "protein": "Aconitate hydratase, mitochondrial",
        "enzyme_role": "enzyme",
        "ec": "4.2.1.3",
        "reaction": "Citrate_to_Isocitrate",
        "reaction_label": "Citrate ↔ Isocitrate",
        "substrates": ["Citrate"],
        "products": ["Isocitrate"],
        "themes": ["TCA/Pyruvate"],
    },
    {
        "gene": "PKM",
        "protein": "Pyruvate kinase PKM",
        "enzyme_role": "enzyme",
        "ec": "2.7.1.40",
        "reaction": "PEP_to_Pyruvate",
        "reaction_label": "Phosphoenolpyruvate → Pyruvate",
        "substrates": ["Phosphoenolpyruvate"],
        "products": ["Pyruvate"],
        "themes": ["Glycolysis/Glucose", "TCA/Pyruvate"],
    },
    {
        "gene": "IDH3G",
        "protein": "Isocitrate dehydrogenase [NAD] subunit gamma, mitochondrial",
        "enzyme_role": "complex-associated",
        "ec": "1.1.1.41",
        "reaction": "Isocitrate_to_AlphaKG_NAD",
        "reaction_label": "Isocitrate → α-Ketoglutarate (IDH3 complex)",
        "substrates": ["Isocitrate"],
        "products": ["Alpha-ketoglutarate"],
        "themes": ["TCA/Pyruvate", "Redox"],
    },
    {
        "gene": "GAPDH",
        "protein": "Glyceraldehyde-3-phosphate dehydrogenase",
        "enzyme_role": "enzyme",
        "ec": "1.2.1.12",
        "reaction": "GAP_to_13BPG",
        "reaction_label": "GAP → 1,3-Bisphosphoglycerate",
        "substrates": ["Glyceraldehyde-3-phosphate"],
        "products": ["1,3-Bisphosphoglycerate"],
        "themes": ["Glycolysis/Glucose", "Redox"],
    },
    {
        "gene": "G6PD",
        "protein": "Glucose-6-phosphate dehydrogenase",
        "enzyme_role": "enzyme",
        "ec": "1.1.1.49",
        "reaction": "G6P_to_6PGL",
        "reaction_label": "Glucose-6-phosphate → 6-Phosphogluconolactone",
        "substrates": ["Glucose-6-phosphate"],
        "products": ["6-Phosphogluconolactone"],
        "themes": ["Glycolysis/Glucose", "Redox/Glutathione", "NAD/One-carbon"],
    },
    {
        "gene": "NDUFV2",
        "protein": "NADH dehydrogenase [ubiquinone] flavoprotein 2, mitochondrial",
        "enzyme_role": "complex-associated",
        "ec": "7.1.1.2",
        "reaction": "Complex_I_NADH_Ubiquinone",
        "reaction_label": "NADH + Ubiquinone → NAD+ + Ubiquinol (Complex I)",
        "substrates": ["NADH", "Ubiquinone"],
        "products": ["NAD+", "Ubiquinol"],
        "themes": ["Mito/OXPHOS", "Redox"],
    },
    {
        "gene": "IDH2",
        "protein": "Isocitrate dehydrogenase [NADP], mitochondrial",
        "enzyme_role": "enzyme",
        "ec": "1.1.1.42",
        "reaction": "Isocitrate_to_AlphaKG_NADP",
        "reaction_label": "Isocitrate → α-Ketoglutarate (IDH2)",
        "substrates": ["Isocitrate"],
        "products": ["Alpha-ketoglutarate"],
        "themes": ["TCA/Pyruvate", "Redox"],
    },
]

curated = pd.DataFrame(curated_records)

display(
    curated[
        [
            "gene",
            "protein",
            "enzyme_role",
            "ec",
            "reaction_label",
        ]
    ]
)

## 6. Verificar cobertura de la anotación

In [ ]:
curated_genes = set(
    curated["gene"]
    .astype(str)
    .str.upper()
)

input_genes = set(
    pd.Series(network_genes)
    .astype(str)
    .str.upper()
)

missing_curated = sorted(
    input_genes - curated_genes
)

coverage = (
    len(input_genes & curated_genes)
    /
    len(input_genes)
)

print(
    f"Curated mapping coverage: "
    f"{len(input_genes & curated_genes)}/{len(input_genes)} "
    f"({coverage:.1%})"
)

if missing_curated:
    print("Genes without curated mapping:")
    print(missing_curated)

if coverage < 0.70:
    raise RuntimeError(
        "Curated mechanistic coverage is <70%."
    )

print("✓ Sufficient curated mapping coverage.")

## 7. Opcional: consultar MyGene

In [ ]:
mygene_annotations = None

try:
    import mygene

    mg = mygene.MyGeneInfo()

    query = mg.querymany(
        network_genes,
        scopes="symbol",
        fields="symbol,name,entrezgene,ensembl.gene,uniprot.Swiss-Prot",
        species="human",
        as_dataframe=True,
        returnall=False,
        verbose=False,
    )

    if isinstance(query, pd.DataFrame):
        mygene_annotations = query.reset_index()

        mygene_annotations.to_csv(
            NB09_RESULTS /
            "mygene_annotations.csv",
            index=False,
        )

        print("✓ MyGene annotations retrieved.")
        display(mygene_annotations.head(20))

except Exception as exc:
    print("MyGene annotation skipped or unavailable.")
    print("Reason:", exc)

## 8. Integrar evidencia transcriptómica y anotación bioquímica

In [ ]:
gene_table = candidates.copy()

gene_table["gene_upper"] = (
    gene_table["signature_gene"]
    .astype(str)
    .str.upper()
)

curated_merge = curated.copy()

curated_merge["gene_upper"] = (
    curated_merge["gene"]
    .astype(str)
    .str.upper()
)

gene_table = gene_table.merge(
    curated_merge,
    on="gene_upper",
    how="left",
)

print("Integrated gene table:", gene_table.shape)

display(
    gene_table[
        [
            "signature_gene",
            "cross_dataset_score",
            "final_validation_class",
            "protein",
            "enzyme_role",
            "ec",
            "reaction_label",
        ]
    ]
)

## 9. Cargar evidencia por tipo celular del Notebook 08

In [ ]:
celltype_evidence = None

if NB08_EVIDENCE.exists():

    celltype_evidence = pd.read_csv(
        NB08_EVIDENCE
    )

    print(
        "Cell-type evidence shape:",
        celltype_evidence.shape,
    )

    display(
        celltype_evidence.head(20)
    )

else:

    print(
        "Notebook 08 cell-type evidence table not found. "
        "Core network can continue."
    )

## 10. Construir tablas de nodos y aristas

In [ ]:
node_rows = []
edge_rows = []

for _, row in gene_table.iterrows():

    gene = str(row["signature_gene"])

    node_rows.append({
        "node_id": f"GENE::{gene}",
        "label": gene,
        "node_type": "gene",
        "gene": gene,
        "cross_dataset_score": row["cross_dataset_score"],
        "validation_class": row["final_validation_class"],
    })

    if pd.notna(row.get("protein")):

        protein = str(row["protein"])

        node_rows.append({
            "node_id": f"PROTEIN::{gene}",
            "label": protein,
            "node_type": "protein",
            "gene": gene,
            "cross_dataset_score": row["cross_dataset_score"],
            "validation_class": row["final_validation_class"],
        })

        edge_rows.append({
            "source": f"GENE::{gene}",
            "target": f"PROTEIN::{gene}",
            "edge_type": "encodes",
            "gene": gene,
        })

    if pd.notna(row.get("reaction")):

        reaction_id = str(row["reaction"])
        reaction_label = str(row["reaction_label"])

        node_rows.append({
            "node_id": f"REACTION::{reaction_id}",
            "label": reaction_label,
            "node_type": "reaction",
            "gene": None,
            "cross_dataset_score": np.nan,
            "validation_class": None,
        })

        edge_rows.append({
            "source": f"PROTEIN::{gene}",
            "target": f"REACTION::{reaction_id}",
            "edge_type": "catalyzes_or_associates",
            "gene": gene,
        })

        substrates = (
            row["substrates"]
            if isinstance(row["substrates"], list)
            else []
        )

        products = (
            row["products"]
            if isinstance(row["products"], list)
            else []
        )

        for metabolite in substrates:

            node_rows.append({
                "node_id": f"METABOLITE::{metabolite}",
                "label": metabolite,
                "node_type": "metabolite",
                "gene": None,
                "cross_dataset_score": np.nan,
                "validation_class": None,
            })

            edge_rows.append({
                "source": f"METABOLITE::{metabolite}",
                "target": f"REACTION::{reaction_id}",
                "edge_type": "substrate_of",
                "gene": gene,
            })

        for metabolite in products:

            node_rows.append({
                "node_id": f"METABOLITE::{metabolite}",
                "label": metabolite,
                "node_type": "metabolite",
                "gene": None,
                "cross_dataset_score": np.nan,
                "validation_class": None,
            })

            edge_rows.append({
                "source": f"REACTION::{reaction_id}",
                "target": f"METABOLITE::{metabolite}",
                "edge_type": "produces",
                "gene": gene,
            })

        themes = (
            row["themes"]
            if isinstance(row["themes"], list)
            else []
        )

        for theme in themes:

            node_rows.append({
                "node_id": f"THEME::{theme}",
                "label": theme,
                "node_type": "theme",
                "gene": None,
                "cross_dataset_score": np.nan,
                "validation_class": None,
            })

            edge_rows.append({
                "source": f"REACTION::{reaction_id}",
                "target": f"THEME::{theme}",
                "edge_type": "belongs_to_theme",
                "gene": gene,
            })

nodes = (
    pd.DataFrame(node_rows)
    .drop_duplicates("node_id")
    .reset_index(drop=True)
)

edges = (
    pd.DataFrame(edge_rows)
    .drop_duplicates(
        ["source", "target", "edge_type"]
    )
    .reset_index(drop=True)
)

print("Nodes:", len(nodes))
print("Edges:", len(edges))

display(nodes.head(30))
display(edges.head(30))

## 11. Exportar tablas de nodos y aristas

In [ ]:
nodes.to_csv(
    NB09_RESULTS /
    "gene_protein_enzyme_reaction_nodes.csv",
    index=False,
)

edges.to_csv(
    NB09_RESULTS /
    "gene_protein_enzyme_reaction_edges.csv",
    index=False,
)

print("✓ Node and edge tables exported.")

## 12. Construir NetworkX graph

In [ ]:
G = nx.DiGraph()

for _, row in nodes.iterrows():
    attrs = row.drop(labels=["node_id"]).to_dict()
    G.add_node(row["node_id"], **attrs)

for _, row in edges.iterrows():
    attrs = row.drop(labels=["source", "target"]).to_dict()
    G.add_edge(row["source"], row["target"], **attrs)

print(G)
print(
    "Connected components (undirected view):",
    nx.number_connected_components(
        G.to_undirected()
    ),
)

## 13. Métricas topológicas

In [ ]:
degree = dict(
    G.to_undirected().degree()
)

betweenness = nx.betweenness_centrality(
    G.to_undirected()
)

centrality_rows = []

for node in G.nodes():

    centrality_rows.append({
        "node_id": node,
        "label": G.nodes[node].get("label"),
        "node_type": G.nodes[node].get("node_type"),
        "degree": degree.get(node, 0),
        "betweenness": betweenness.get(node, 0.0),
    })

centrality = (
    pd.DataFrame(centrality_rows)
    .sort_values(
        [
            "betweenness",
            "degree",
        ],
        ascending=False,
    )
)

centrality.to_csv(
    NB09_RESULTS /
    "network_centrality.csv",
    index=False,
)

display(
    centrality.head(30)
)

## 14. Resumen de genes por tema metabólico

In [ ]:
theme_rows = []

for _, row in gene_table.iterrows():

    themes = (
        row["themes"]
        if isinstance(row["themes"], list)
        else []
    )

    for theme in themes:

        theme_rows.append({
            "gene": row["signature_gene"],
            "theme": theme,
            "cross_dataset_score": row["cross_dataset_score"],
            "validation_class": row["final_validation_class"],
        })

theme_table = pd.DataFrame(theme_rows)

if len(theme_table):

    theme_summary = (
        theme_table
        .groupby(
            "theme",
            observed=True,
        )
        .agg(
            n_genes=("gene", "nunique"),
            mean_cross_dataset_score=(
                "cross_dataset_score",
                "mean",
            ),
        )
        .reset_index()
        .sort_values(
            [
                "n_genes",
                "mean_cross_dataset_score",
            ],
            ascending=[
                False,
                False,
            ],
        )
    )

    theme_summary.to_csv(
        NB09_RESULTS /
        "network_theme_summary.csv",
        index=False,
    )

    display(theme_summary)

## 15. Identificar genes conectados por metabolitos compartidos

In [ ]:
gene_reaction_map = {}

for _, row in gene_table.iterrows():

    gene = str(row["signature_gene"])

    if not isinstance(row.get("substrates"), list):
        continue

    metabolites = set(
        row["substrates"]
        +
        row["products"]
    )

    gene_reaction_map[gene] = metabolites

shared_metabolite_rows = []

genes = sorted(
    gene_reaction_map.keys()
)

for i, g1 in enumerate(genes):

    for g2 in genes[i + 1:]:

        shared = (
            gene_reaction_map[g1]
            &
            gene_reaction_map[g2]
        )

        if shared:

            shared_metabolite_rows.append({
                "gene_1": g1,
                "gene_2": g2,
                "shared_metabolites": "; ".join(
                    sorted(shared)
                ),
                "n_shared": len(shared),
            })

shared_metabolites = pd.DataFrame(
    shared_metabolite_rows
)

shared_metabolites.to_csv(
    NB09_RESULTS /
    "gene_pairs_shared_metabolites.csv",
    index=False,
)

display(shared_metabolites)

## 16. Construir red gen–gen simplificada

In [ ]:
G_gene = nx.Graph()

candidate_lookup = (
    candidates
    .set_index("signature_gene")
)

for gene in network_genes:

    score = float(
        candidate_lookup.loc[
            gene,
            "cross_dataset_score",
        ]
    )

    validation_class = str(
        candidate_lookup.loc[
            gene,
            "final_validation_class",
        ]
    )

    G_gene.add_node(
        gene,
        score=score,
        validation_class=validation_class,
    )

for _, row in shared_metabolites.iterrows():

    g1 = row["gene_1"]
    g2 = row["gene_2"]

    G_gene.add_edge(
        g1,
        g2,
        shared_metabolite=True,
        shared_theme=False,
        same_reaction=False,
    )

gene_themes = {
    str(row["signature_gene"]): set(
        row["themes"]
        if isinstance(row["themes"], list)
        else []
    )
    for _, row in gene_table.iterrows()
}

for i, g1 in enumerate(network_genes):

    for g2 in network_genes[i + 1:]:

        if g1 not in gene_themes or g2 not in gene_themes:
            continue

        shared_themes = (
            gene_themes[g1]
            &
            gene_themes[g2]
        )

        if shared_themes:

            if G_gene.has_edge(g1, g2):
                G_gene[g1][g2]["shared_theme"] = True
            else:
                G_gene.add_edge(
                    g1,
                    g2,
                    shared_metabolite=False,
                    shared_theme=True,
                    same_reaction=False,
                )

reaction_gene_groups = (
    gene_table
    .dropna(subset=["reaction"])
    .groupby("reaction")["signature_gene"]
    .apply(list)
)

for reaction_id, reaction_genes in reaction_gene_groups.items():

    if len(reaction_genes) < 2:
        continue

    for i, g1 in enumerate(reaction_genes):
        for g2 in reaction_genes[i + 1:]:

            if G_gene.has_edge(g1, g2):
                G_gene[g1][g2]["same_reaction"] = True
            else:
                G_gene.add_edge(
                    g1,
                    g2,
                    shared_metabolite=False,
                    shared_theme=False,
                    same_reaction=True,
                )

print(G_gene)

## 17. Exportar red gen–gen

In [ ]:
gene_gene_edges = []

for u, v, data in G_gene.edges(data=True):

    gene_gene_edges.append({
        "gene_1": u,
        "gene_2": v,
        "shared_metabolite": data.get(
            "shared_metabolite",
            False,
        ),
        "shared_theme": data.get(
            "shared_theme",
            False,
        ),
        "same_reaction": data.get(
            "same_reaction",
            False,
        ),
    })

gene_gene_edges = pd.DataFrame(
    gene_gene_edges
)

gene_gene_edges.to_csv(
    NB09_RESULTS /
    "gene_gene_network_edges.csv",
    index=False,
)

display(gene_gene_edges)

## 18. Visualizar red gen–gen

In [ ]:
if G_gene.number_of_nodes() > 0:

    pos = nx.spring_layout(
        G_gene,
        seed=RANDOM_STATE,
        k=1.2,
    )

    scores = np.array([
        G_gene.nodes[n].get(
            "score",
            0.0,
        )
        for n in G_gene.nodes()
    ])

    node_sizes = (
        700
        +
        2200
        *
        scores
    )

    fig, ax = plt.subplots(
        figsize=(11, 9)
    )

    nx.draw_networkx_edges(
        G_gene,
        pos,
        ax=ax,
        width=1.0,
        alpha=0.45,
    )

    nx.draw_networkx_nodes(
        G_gene,
        pos,
        ax=ax,
        node_size=node_sizes,
        alpha=0.85,
    )

    nx.draw_networkx_labels(
        G_gene,
        pos,
        ax=ax,
        font_size=9,
        font_weight="bold",
    )

    ax.set_title(
        "Validated metabolic gene network"
    )

    ax.axis("off")

    fig.tight_layout()

    fig.savefig(
        NB09_FIGURES /
        "validated_metabolic_gene_network.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

## 19. Visualizar red completa

In [ ]:
if G.number_of_nodes() > 0:

    pos = nx.spring_layout(
        G.to_undirected(),
        seed=RANDOM_STATE,
        k=0.9,
        iterations=200,
    )

    node_type_size = {
        "gene": 900,
        "protein": 450,
        "reaction": 550,
        "metabolite": 350,
        "theme": 700,
    }

    fig, ax = plt.subplots(
        figsize=(16, 13)
    )

    nx.draw_networkx_edges(
        G,
        pos,
        ax=ax,
        arrows=True,
        arrowsize=10,
        width=0.8,
        alpha=0.28,
    )

    for node_type in [
        "gene",
        "protein",
        "reaction",
        "metabolite",
        "theme",
    ]:

        nodelist = [
            n
            for n, d in G.nodes(data=True)
            if d.get("node_type") == node_type
        ]

        if not nodelist:
            continue

        nx.draw_networkx_nodes(
            G,
            pos,
            nodelist=nodelist,
            node_size=node_type_size[
                node_type
            ],
            alpha=0.78,
            ax=ax,
            label=node_type,
        )

    labels = {}

    for n, d in G.nodes(data=True):

        if d.get("node_type") in {
            "gene",
            "metabolite",
            "theme",
        }:

            labels[n] = d.get(
                "label",
                n,
            )

    nx.draw_networkx_labels(
        G,
        pos,
        labels=labels,
        font_size=7,
        ax=ax,
    )

    ax.legend(
        frameon=False,
        loc="upper left",
    )

    ax.set_title(
        "Gene–protein–reaction–metabolite network"
    )

    ax.axis("off")

    fig.tight_layout()

    fig.savefig(
        NB09_FIGURES /
        "gene_protein_reaction_metabolite_network.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

## 20. Ranking mecanístico integrado

In [ ]:
gene_degree = dict(
    G_gene.degree()
)

integrated_rows = []

for _, row in gene_table.iterrows():

    gene = row["signature_gene"]

    themes = (
        row["themes"]
        if isinstance(row["themes"], list)
        else []
    )

    integrated_rows.append({
        "gene": gene,
        "cross_dataset_score": row[
            "cross_dataset_score"
        ],
        "validation_class": row[
            "final_validation_class"
        ],
        "gene_network_degree": gene_degree.get(
            gene,
            0,
        ),
        "n_themes": len(themes),
        "has_reaction_annotation": pd.notna(
            row["reaction"]
        ),
        "enzyme_role": row[
            "enzyme_role"
        ],
    })

integrated_rank = pd.DataFrame(
    integrated_rows
)

if integrated_rank[
    "gene_network_degree"
].max() > 0:

    integrated_rank[
        "degree_score"
    ] = (
        integrated_rank[
            "gene_network_degree"
        ]
        /
        integrated_rank[
            "gene_network_degree"
        ].max()
    )

else:

    integrated_rank[
        "degree_score"
    ] = 0.0

if integrated_rank[
    "n_themes"
].max() > 0:

    integrated_rank[
        "theme_score"
    ] = (
        integrated_rank[
            "n_themes"
        ]
        /
        integrated_rank[
            "n_themes"
        ].max()
    )

else:

    integrated_rank[
        "theme_score"
    ] = 0.0

integrated_rank[
    "mechanistic_priority_score"
] = (
    0.70
    *
    integrated_rank[
        "cross_dataset_score"
    ]
    +
    0.20
    *
    integrated_rank[
        "degree_score"
    ]
    +
    0.10
    *
    integrated_rank[
        "theme_score"
    ]
)

integrated_rank = integrated_rank.sort_values(
    "mechanistic_priority_score",
    ascending=False,
)

integrated_rank[
    "mechanistic_rank"
] = np.arange(
    1,
    len(integrated_rank) + 1,
)

integrated_rank.to_csv(
    NB09_RESULTS /
    "mechanistic_gene_priority.csv",
    index=False,
)

display(integrated_rank)

## 21. Resumen de localización celular

In [ ]:
celltype_gene_summary = None

if (
    celltype_evidence is not None
    and
    len(celltype_evidence) > 0
):

    tmp = celltype_evidence.copy()

    tmp["sn_padj"] = pd.to_numeric(
        tmp["sn_padj"],
        errors="coerce",
    )

    tmp["sn_log2FC"] = pd.to_numeric(
        tmp["sn_log2FC"],
        errors="coerce",
    )

    rows = []

    for gene, sub in tmp.groupby(
        "signature_gene",
        observed=True,
    ):

        sig_sub = sub.loc[
            sub["sn_padj"].notna()
        ].copy()

        if len(sig_sub):

            best = sig_sub.sort_values(
                [
                    "sn_padj",
                    "sn_log2FC",
                ],
                ascending=[
                    True,
                    True,
                ],
            ).iloc[0]

            best_celltype = best[
                "cell_type"
            ]

            best_padj = best[
                "sn_padj"
            ]

            best_lfc = best[
                "sn_log2FC"
            ]

        else:

            best_celltype = None
            best_padj = np.nan
            best_lfc = np.nan

        significant_celltypes = (
            sub.loc[
                sub["sn_FDR_lt_0.05"]
                ==
                True,
                "cell_type",
            ]
            .astype(str)
            .tolist()
        )

        rows.append({
            "gene": gene,
            "best_celltype_by_padj": best_celltype,
            "best_sn_padj": best_padj,
            "best_sn_log2FC": best_lfc,
            "significant_celltypes": "; ".join(
                significant_celltypes
            ),
            "n_significant_celltypes": len(
                significant_celltypes
            ),
        })

    celltype_gene_summary = pd.DataFrame(
        rows
    )

    celltype_gene_summary.to_csv(
        NB09_RESULTS /
        "gene_celltype_localization_summary.csv",
        index=False,
    )

    display(celltype_gene_summary)

## 22. Tabla final integrada

In [ ]:
final_gene_table = gene_table.copy()

final_gene_table = final_gene_table.merge(
    integrated_rank[
        [
            "gene",
            "gene_network_degree",
            "n_themes",
            "mechanistic_priority_score",
            "mechanistic_rank",
        ]
    ],
    left_on="signature_gene",
    right_on="gene",
    how="left",
)

if celltype_gene_summary is not None:

    final_gene_table = final_gene_table.merge(
        celltype_gene_summary,
        left_on="signature_gene",
        right_on="gene",
        how="left",
        suffixes=("", "_celltype"),
    )

final_gene_table.to_csv(
    NB09_RESULTS /
    "final_gene_protein_enzyme_reaction_table.csv",
    index=False,
)

cols_to_show = [
    c
    for c in [
        "signature_gene",
        "cross_dataset_score",
        "final_validation_class",
        "mechanistic_priority_score",
        "mechanistic_rank",
        "protein",
        "enzyme_role",
        "ec",
        "reaction_label",
        "best_celltype_by_padj",
        "n_significant_celltypes",
    ]
    if c in final_gene_table.columns
]

display(
    final_gene_table[
        cols_to_show
    ].sort_values(
        "mechanistic_rank"
    )
)

## 23. Exportar GraphML

In [ ]:
G_export = nx.DiGraph()

for node, data in G.nodes(data=True):

    clean_data = {}

    for key, value in data.items():

        if isinstance(value, (list, dict, tuple, set)):
            clean_data[key] = str(value)
        elif value is None:
            clean_data[key] = ""
        elif isinstance(value, float) and np.isnan(value):
            clean_data[key] = ""
        elif isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
                np.integer,
                np.floating,
            ),
        ):
            clean_data[key] = value
        else:
            clean_data[key] = str(value)

    G_export.add_node(
        node,
        **clean_data,
    )

for u, v, data in G.edges(data=True):

    clean_data = {}

    for key, value in data.items():

        if isinstance(value, (list, dict, tuple, set)):
            clean_data[key] = str(value)
        elif value is None:
            clean_data[key] = ""
        elif isinstance(value, float) and np.isnan(value):
            clean_data[key] = ""
        elif isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
                np.integer,
                np.floating,
            ),
        ):
            clean_data[key] = value
        else:
            clean_data[key] = str(value)

    G_export.add_edge(
        u,
        v,
        **clean_data,
    )

graphml_path = (
    NB09_RESULTS /
    "gene_protein_enzyme_reaction_network.graphml"
)

nx.write_graphml(
    G_export,
    graphml_path,
)

print("✓ GraphML exported:")
print(graphml_path.resolve())

## 24. Exportar input para Notebook 10 — Machine Learning

In [ ]:
ml_input = final_gene_table[
    [
        c
        for c in [
            "signature_gene",
            "cross_dataset_score",
            "final_validation_class",
            "mechanistic_priority_score",
            "mechanistic_rank",
            "bulk_log2FC",
            "bulk_padj",
            "median_sn_log2FC",
            "enzyme_role",
        ]
        if c in final_gene_table.columns
    ]
].copy()

ml_input.to_csv(
    NB09_RESULTS /
    "Notebook10_ML_feature_candidates.csv",
    index=False,
)

display(ml_input)

## 25. Manifest de Notebook 09

In [ ]:
manifest09 = {
    "input": str(NETWORK_INPUT),
    "n_genes": int(
        len(network_genes)
    ),
    "n_full_network_nodes": int(
        G.number_of_nodes()
    ),
    "n_full_network_edges": int(
        G.number_of_edges()
    ),
    "n_gene_network_nodes": int(
        G_gene.number_of_nodes()
    ),
    "n_gene_network_edges": int(
        G_gene.number_of_edges()
    ),
    "curated_mapping_coverage": float(
        coverage
    ),
    "network_layers": [
        "gene",
        "protein",
        "reaction",
        "metabolite",
        "theme",
    ],
    "mechanistic_score": (
        "70% cross-dataset score + "
        "20% gene-network degree + "
        "10% metabolic-theme count"
    ),
    "mechanistic_score_is_probability": False,
    "graphml": str(
        graphml_path
    ),
    "next_notebook": (
        "10_machine_learning.ipynb"
    ),
    "interpretation_warning": (
        "Transcript-level evidence and curated biochemical "
        "relationships do not directly establish protein abundance, "
        "enzyme activity, causal mechanism, or metabolic flux."
    ),
}

manifest09_path = (
    NB09_RESULTS /
    "notebook09_manifest.json"
)

with open(
    manifest09_path,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        manifest09,
        f,
        indent=2,
        ensure_ascii=False,
    )

print(
    json.dumps(
        manifest09,
        indent=2,
        ensure_ascii=False,
    )
)

## 26. Checkpoint final

In [ ]:
checks09 = {
    "Notebook08 candidates loaded":
        len(candidates) > 0,

    "At least 3 genes available":
        len(network_genes) >= 3,

    "Curated mechanistic coverage >=70%":
        coverage >= 0.70,

    "Gene table generated":
        len(gene_table) > 0,

    "Node table generated":
        len(nodes) > 0,

    "Edge table generated":
        len(edges) > 0,

    "Full NetworkX graph generated":
        G.number_of_nodes() > 0,

    "Gene-gene network generated":
        G_gene.number_of_nodes() > 0,

    "Mechanistic ranking generated":
        len(integrated_rank) > 0,

    "Final integrated table exported":
        (
            NB09_RESULTS /
            "final_gene_protein_enzyme_reaction_table.csv"
        ).exists(),

    "GraphML exported":
        graphml_path.exists(),

    "Notebook10 feature table exported":
        (
            NB09_RESULTS /
            "Notebook10_ML_feature_candidates.csv"
        ).exists(),

    "Manifest exported":
        manifest09_path.exists(),
}

checkpoint09 = (
    pd.Series(
        checks09,
        name="passed",
    )
    .to_frame()
)

checkpoint09[
    "status"
] = (
    checkpoint09[
        "passed"
    ]
    .map({
        True: "PASS",
        False: "FAIL",
    })
)

display(checkpoint09)

checkpoint_path = (
    NB09_RESULTS /
    "notebook09_final_checkpoint.csv"
)

checkpoint09.to_csv(
    checkpoint_path
)

failed = checkpoint09.loc[
    ~checkpoint09[
        "passed"
    ]
]

if len(failed):

    print(
        "\nSome Notebook 09 checks failed:"
    )

    display(failed)

    print(
        "\nNotebook 10 should not be started yet."
    )

else:

    print(
        "\n✓ All Notebook 09 checks passed."
    )

    print(
        "\nNotebook 09 completed successfully."
    )

    print(
        "\nNext:"
    )

    print(
        "10_machine_learning.ipynb"
    )

## 27. Limitaciones

1. La anotación reacción/metabolito es una representación mecanística curada y simplificada.
2. Algunos genes codifican subunidades de complejos, no enzimas independientes.
3. La red no cuantifica estequiometría ni flujo.
4. `cross_dataset_score` y `mechanistic_priority_score` son scores de priorización, no probabilidades.
5. La centralidad depende de cómo se construyó la red y no implica importancia causal.
6. Los resultados transcriptómicos requieren validación experimental antes de inferir cambios bioquímicos.